# 01 — Exploração dos dados (Fase 1)

Objetivo: medir o que os dados dizem **antes** de escolher qualquer técnica. Cada seção responde a uma pergunta e termina com a decisão que ela justifica.

Antes de abrir este notebook, rode da raiz do repositório:

```
python scripts/run_fase1.py
```

O script grava tabelas e figuras em `results/fase1/`; aqui só lemos e interpretamos.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from pdi_defeitos.config import load_config

cfg = load_config()
res = cfg['root'] / 'results' / 'fase1'
amostras = pd.read_csv(res / 'amostras.csv')
resumo = pd.read_csv(res / 'resumo_por_tipo.csv', index_col=0)
radial = pd.read_csv(res / 'variabilidade_radial.csv')
amostras.groupby(['split', 'defect_type']).size().rename('n').to_frame()

## Galeria: 5 exemplos por tipo, contorno da máscara GT em vermelho

In [ ]:
display(Image(res / 'galeria.png'))

## Pergunta 1 — Existe ruído de sensor para o denoising remover?

Estimador de Immerkær (diferença de Laplacianos, insensível a bordas) na resolução original 900×900.

In [ ]:
amostras['noise_sigma_full'].describe().round(2)

**Leitura:** σ abaixo de 1 nível de cinza em todas as imagens. Ruído de sensor é desprezível.

**Decisão:** o denoising não tem a missão clássica de remover ruído. A missão passa a ser reduzir a variação espúria entre a imagem e a referência (Perguntas 2 e 3).

## Pergunta 2 — A garrafa está sempre na mesma posição?

Centro = centroide da máscara de Otsu (garrafa escura × fundo branco), imagens a 256×256.

In [ ]:
train = amostras[amostras.split == 'train']
pd.DataFrame({'desvio_px': train[['center_x', 'center_y']].std(),
              'amplitude_px': train[['center_x', 'center_y']].max() - train[['center_x', 'center_y']].min()}).round(2)

**Leitura:** o centro varia pouco (desvio < 1 px), mas a amplitude chega a ~3 px. Nas bordas nítidas dos anéis, 3 px bastam para criar diferença grande para a imagem média.

**Decisão:** alinhar por translação antes de montar a referência (Fase 3), e medir se o desvio por pixel nos anéis cai depois disso.

## Pergunta 3 — Onde a garrafa boa varia naturalmente?

Desvio padrão por pixel entre as 209 boas do treino, resumido por faixa de raio (0 = centro, 1 = borda da garrafa).

In [ ]:
display(Image(res / 'variabilidade.png'))
radial.round(2)

**Leitura:** o fundo da garrafa (raio < 0,5) varia 1 a 3 níveis; os anéis (raio > 0,5) variam 9 a 17 níveis, com pontos de reflexo acima de 50.

**Decisão:** normalizar a diferença pelo desvio por pixel (mapa z), para dar menos peso aos anéis; esperar falsos positivos concentrados nos anéis e nos reflexos.

## Pergunta 4 — Como são os defeitos: tamanho, posição e contraste?

- `area_px`, `equiv_diameter_px`: tamanho a 256×256
- `radial_pos`: distância do centroide do defeito ao centro, dividida pelo raio da garrafa
- `dL`: brilho (L do Lab) do defeito menos o de um anel ao redor; `dab`: mudança de cor no plano (a, b)

In [ ]:
cols = ['n', 'area_px_min', 'area_px_median', 'equiv_diameter_px_min', 'n_components_median',
        'radial_pos_median', 'dL_median', 'dab_median']
resumo[cols].round(2)

**Leitura:**

- Menor defeito: ~380 px (contamination) e ~580 px (broken_small), diâmetro equivalente ≥ 22 px.
- Quase sempre 1 componente por máscara.
- broken fica nos anéis (raio ~0,4 a 0,6) e é mais claro que a vizinhança (vidro fraturado reflete); contamination fica no fundo (raio ~0,1).
- A mudança de cor mediana é pequena para todos os tipos, embora algumas contaminações sejam claramente coloridas (veja a galeria).

**Decisões:**

1. Tamanho de filtro, raio do elemento estruturante e área mínima de componente ficam bem abaixo do menor defeito (A_min < ~380 px).
2. A posição radial entra como atributo geométrico: separa broken de contamination.
3. Segmentar só por cor é arriscado; o método B usa também textura e posição.